In [ ]:
import os, sys, math, json
import numpy as np
import torch
import warnings
warnings.filterwarnings("ignore")

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"

SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

CANDIDATES = [
    "/kaggle/input/smart-mcq-solver-challenge",
    os.environ.get("MCQ_DATA_DIR", ""),
    ".",
    "./data",
]
DATA_DIR = next((p for p in CANDIDATES if p and os.path.exists(os.path.join(p, "train.csv"))), None)
assert DATA_DIR is not None, "train.csv not found. Set MCQ_DATA_DIR env var."
TRAIN_CSV = os.path.join(DATA_DIR, "train.csv")
print(f"DATA_DIR = {DATA_DIR}")
print(f"TRAIN_CSV exists: {os.path.exists(TRAIN_CSV)}")

ANSWERS = {}


In [ ]:
from datasets import load_dataset

ds = load_dataset("csv", data_files=TRAIN_CSV, split="train")

def add_combined(example):
    return {"combined_text": example["prompt"] + " " + example["A"]}

ds = ds.map(add_combined)
Q1 = len(ds[51]["combined_text"])
print(f"Q1: len(combined_text[51]) = {Q1}")
ANSWERS["Q1"] = Q1


In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
Q2 = tokenizer.vocab_size
print(f"Q2: vocab_size = {Q2}")
ANSWERS["Q2"] = Q2


In [ ]:
Q3 = tokenizer.sep_token_id
print(f"Q3: [SEP] token id = {Q3}  (sep_token = {tokenizer.sep_token!r})")
ANSWERS["Q3"] = Q3


In [ ]:
prompts = list(ds["prompt"])
encoded = tokenizer(
    prompts,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt",
)
Q4 = tuple(encoded["input_ids"].shape)
print(f"Q4: input_ids.shape = {Q4}")
ANSWERS["Q4"] = Q4


In [ ]:
HIDDEN_SIZE = 768
NUM_HEADS = 12
Q5 = HIDDEN_SIZE // NUM_HEADS
print(f"Q5: per-head dim = {HIDDEN_SIZE} / {NUM_HEADS} = {Q5}")
ANSWERS["Q5"] = Q5


In [ ]:
from transformers import AutoModel

model = AutoModel.from_pretrained("bert-base-uncased")
model.eval()

row0_prompt = ds[0]["prompt"]
inputs = tokenizer(row0_prompt, return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs)

Q6 = tuple(outputs.last_hidden_state.shape)
print(f"Q6: last_hidden_state.shape = {Q6}")
ANSWERS["Q6"] = Q6


In [ ]:
cls_vec = outputs.last_hidden_state[0, 0, :]
first5 = cls_vec[:5]
Q7 = round(first5.sum().item(), 4)
print(f"Q7: sum(first 5 CLS values) = {Q7}")
print(f"raw first-5 values: {first5.tolist()}")
ANSWERS["Q7"] = Q7


In [ ]:
model_attn = AutoModel.from_pretrained("bert-base-uncased", output_attentions=True)
model_attn.eval()

text = "Light-ion fusion is a technique."
enc = tokenizer(text, return_tensors="pt")
tokens = tokenizer.convert_ids_to_tokens(enc["input_ids"][0].tolist())
print(f"Tokens: {tokens}")

fusion_idx = tokens.index("fusion")
cls_idx = tokens.index("[CLS]")
print(f"[CLS] @ {cls_idx}, 'fusion' @ {fusion_idx}")

with torch.no_grad():
    out = model_attn(**enc)

attn_last_h0 = out.attentions[-1][0, 0]
cls_to_fusion = attn_last_h0[cls_idx, fusion_idx].item()
Q8 = round(cls_to_fusion, 4)
print(f"Q8: attn([CLS] -> 'fusion') = {Q8}")
ANSWERS["Q8"] = Q8


In [ ]:
from sentence_transformers import SentenceTransformer, util

st_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

prompt_0 = ds[0]["prompt"]
opt_b_0 = ds[0]["B"]

emb_prompt = st_model.encode(prompt_0, convert_to_tensor=True)
emb_b = st_model.encode(opt_b_0, convert_to_tensor=True)

Q9 = round(util.cos_sim(emb_prompt, emb_b).item(), 4)
print(f"Q9: cos_sim(prompt, B) at row 0 = {Q9}")
ANSWERS["Q9"] = Q9


In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

df = pd.read_csv(TRAIN_CSV)
N = len(df)
OPTS = ["A", "B", "C", "D", "E"]
print(f"Rows: {N}")

def tfidf_top3(row):
    texts = [row["prompt"]] + [row[c] for c in OPTS]
    vec = TfidfVectorizer().fit(texts)
    mat = vec.transform(texts)
    sims = cosine_similarity(mat[0:1], mat[1:]).ravel()
    order = np.argsort(-sims)[:3]
    return [OPTS[i] for i in order]

tfidf_preds = [tfidf_top3(df.iloc[i]) for i in range(N)]
print("TF-IDF done.")


In [ ]:
prompt_embs = st_model.encode(df["prompt"].tolist(), batch_size=64,
                              convert_to_tensor=True, show_progress_bar=True)
option_embs = {}
for c in OPTS:
    option_embs[c] = st_model.encode(df[c].tolist(), batch_size=64,
                                     convert_to_tensor=True, show_progress_bar=True)

def minilm_top3(i):
    p = prompt_embs[i]
    sims = []
    for c in OPTS:
        sims.append(util.cos_sim(p, option_embs[c][i]).item())
    sims = np.array(sims)
    order = np.argsort(-sims)[:3]
    return [OPTS[i_] for i_ in order]

minilm_preds = [minilm_top3(i) for i in range(N)]
print("MiniLM done.")


In [ ]:
def map_at_k(preds, truths, k=3):
    score = 0.0
    for p, t in zip(preds, truths):
        for r, pred in enumerate(p[:k]):
            if pred == t:
                score += 1.0 / (r + 1)
                break
    return score / len(preds)

truths = df["answer"].tolist()
minilm_map3 = map_at_k(minilm_preds, truths, 3)
tfidf_map3 = map_at_k(tfidf_preds, truths, 3)
print(f"TF-IDF  MAP@3: {tfidf_map3:.4f}")
print(f"MiniLM  MAP@3: {minilm_map3:.4f}")
Q10_a = round(minilm_map3, 4)
print(f"Q10 (a): MiniLM MAP@3 = {Q10_a}")
ANSWERS["Q10_MAP3_MiniLM"] = Q10_a

miss_tfidf_hit_minilm = sum(
    1 for i in range(N)
    if truths[i] not in tfidf_preds[i] and truths[i] in minilm_preds[i]
)
Q10_b = miss_tfidf_hit_minilm
print(f"Q10 (b): rows where TF-IDF misses but MiniLM hits = {Q10_b}")
ANSWERS["Q10_TFIDF_miss_MiniLM_hit"] = Q10_b


In [ ]:
from transformers import pipeline

zsc = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

row1 = ds[1]
candidates_11 = [row1["A"], row1["B"], row1["C"]]
res_softmax = zsc(row1["prompt"], candidate_labels=candidates_11)
print("Softmax result:")
for lbl, sc in zip(res_softmax["labels"], res_softmax["scores"]):
    print(f"  {sc:.6f}  {lbl[:80]}")

Q11 = round(res_softmax["scores"][0], 4)
print(f"Q11: top probability = {Q11}")
ANSWERS["Q11"] = Q11


In [ ]:
res_sigmoid = zsc(row1["prompt"], candidate_labels=candidates_11, multi_label=True)
print("Sigmoid result:")
for lbl, sc in zip(res_sigmoid["labels"], res_sigmoid["scores"]):
    print(f"  {sc:.6f}  {lbl[:80]}")

sum_softmax = sum(res_softmax["scores"])
sum_sigmoid = sum(res_sigmoid["scores"])
print(f"sum(softmax) = {sum_softmax:.6f}")
print(f"sum(sigmoid) = {sum_sigmoid:.6f}")

Q12 = round(abs(sum_softmax - sum_sigmoid), 4)
print(f"Q12: |Δ| = {Q12}")
ANSWERS["Q12"] = Q12


In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

t5_tok = AutoTokenizer.from_pretrained("google/flan-t5-small")
t5_model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small")
t5_model.eval()

row0 = ds[0]
prompt_str = (
    f"Question: {row0['prompt']}. Is the correct answer "
    f"A: {row0['A']} or B: {row0['B']}? "
    f"Answer with just the letter A or B."
)
print(prompt_str[:300])

inputs = t5_tok(prompt_str, return_tensors="pt", truncation=True)
with torch.no_grad():
    out_ids = t5_model.generate(
        **inputs,
        max_new_tokens=5,
        do_sample=False,
        num_beams=1,
    )

Q13 = t5_tok.decode(out_ids[0], skip_special_tokens=True)
print(f"Q13: T5 output = {Q13!r}")
ANSWERS["Q13"] = Q13


In [ ]:
for k, v in ANSWERS.items():
    print(f"{k:30s} = {v}")

with open("milestone-2-answers.json", "w") as f:
    json.dump(ANSWERS, f, indent=2, default=str)
print("Saved milestone-2-answers.json")
